# FusionModel Training on PTB‑XL (ECG + Metadata)

## Sections
1. Load PTB‑XL ECG + Metadata
2. Build Fusion Dataloaders
3. Load Pretrained ECG & Metadata Models
4. Build Fusion Model
5. Train Fusion Classifier
6. Training of the fusion model
7. Evaluation of the fusion model
8. Full training
9. Save best model

## 0. Setup

In [1]:
import sys
sys.path.append("C:/Users/inaki/Desktop/TFG/Code/ptbxl_project")

print(sys.executable)


c:\Users\inaki\anaconda3\envs\ptbxl\python.exe


## 1. Load PTB‑XL ECG + Metadata

In [2]:
import pandas as pd
from utils import load_ptbxl
from utils.data_loader import load_metadata

# Path and sampling rate
PATH = "C:/Users/inaki/Desktop/TFG/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"
SAMPLING_RATE = 100

# -------------------------------
# Load ECG signals + labels
# -------------------------------
X_train, X_val, X_test, y_train, y_val, y_test, classes = load_ptbxl(
    PATH, SAMPLING_RATE
)

print("ECG shapes:", X_train.shape, X_val.shape, X_test.shape)

# -------------------------------
# Load metadata CSV
# -------------------------------
meta_full = pd.read_csv(PATH + "ptbxl_database.csv")
meta_full["recording_date"] = pd.to_datetime(meta_full["recording_date"])

# -------------------------------
# Official PTB‑XL fold splits
# -------------------------------
train_folds = list(range(1, 9))   # folds 1–8
val_fold = 9                      # fold 9
test_fold = 10                    # fold 10

# -------------------------------
# Split metadata using folds
# -------------------------------
meta_train = load_metadata(meta_full[meta_full.strat_fold.isin(train_folds)])
meta_val   = load_metadata(meta_full[meta_full.strat_fold == val_fold])
meta_test  = load_metadata(meta_full[meta_full.strat_fold == test_fold])

print("Metadata shapes:", meta_train.shape, meta_val.shape, meta_test.shape)


ECG shapes: (17418, 1000, 12) (2183, 1000, 12) (2198, 1000, 12)
Metadata shapes: (17418, 10) (2183, 10) (2198, 10)


## 2. Build Fusion Dataloaders

In [3]:
from torch.utils.data import DataLoader
from utils.data_loader import FusionDataset

# Reset metadata indices to align with ECG arrays
meta_train_fixed = meta_train.reset_index(drop=True)
meta_val_fixed   = meta_val.reset_index(drop=True)
meta_test_fixed  = meta_test.reset_index(drop=True)

# Build fusion datasets
fusion_train_ds = FusionDataset(X_train, meta_train_fixed, y_train)
fusion_val_ds   = FusionDataset(X_val, meta_val_fixed, y_val)
fusion_test_ds  = FusionDataset(X_test, meta_test_fixed, y_test)

# Build dataloaders
fusion_train_dl = DataLoader(fusion_train_ds, batch_size=32, shuffle=True)
fusion_val_dl   = DataLoader(fusion_val_ds, batch_size=32)
fusion_test_dl  = DataLoader(fusion_test_ds, batch_size=32)

## 3. Load Pretrained ECG & Metadata Models

In [4]:
import torch
from models.meta_mlp import MetaMLP
from models.xresnet1d import xresnet1d101

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load ECG model WITH full head intact (outputs 5-dim logits, no Identity replacement)
ecg_model = xresnet1d101(input_channels=12, num_classes=5)
ecg_model.load_state_dict(torch.load("../outputs/best_ecg_model.pt", map_location=DEVICE))
ecg_model = ecg_model.to(DEVICE)

# Verify — should print (1, 5)
with torch.no_grad():
    sample = torch.randn(1, 12, 1000, device=DEVICE)
    feat = ecg_model(sample)
    print("ecg_feat shape:", feat.shape)  # expected: (1, 5)

meta_model = MetaMLP(in_features=meta_train.shape[1]).to(DEVICE)

C:\Users\inaki\AppData\Local\Temp\ipykernel_11100\2848032710.py:9: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ecg_model.load_state_dict(torch.load("../outputs/best_ecg_mo

ecg_feat shape: torch.Size([1, 5])


## 4. Build Fusion Model

In [5]:
from models.fusion import FusionModel

num_classes = len(classes)   # 5

fusion_model = FusionModel(
    ecg_model  = ecg_model,
    meta_model = meta_model,
    num_classes= num_classes
).to(DEVICE)

# Verify fusion dim — should print 37
print("Fusion dim:", fusion_model.classifier[0].weight.shape[1])  # 37
print(f"Total parameters: {sum(p.numel() for p in fusion_model.parameters())}")

Fusion dim: 37
Total parameters: 1811498


## 5. Train Fusion Classifier

In [ ]:
import importlib
import models.fusion as fusion_module
importlib.reload(fusion_module)

from models.fusion import FusionModel

# RECREATE MODEL
fusion_model = FusionModel(
    ecg_model=ecg_model,
    meta_model=meta_model,
    num_classes=num_classes
).to(DEVICE)

import torch.nn as nn

criterion = nn.BCEWithLogitsLoss()

# Freeze ECG model (recommended)
for param in fusion_model.ecg.parameters():
    param.requires_grad = False

optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-4)

EPOCHS = 20
PATIENCE = 5   # stop if no improvement for 5 epochs

best_val_loss = float("inf")
epochs_no_improve = 0

for epoch in range(EPOCHS):
    # ---------------------
    # TRAIN
    # ---------------------
    fusion_model.train()
    total_loss = 0

    for ecg, meta, y in fusion_train_dl:
        ecg, meta, y = ecg.to(DEVICE), meta.to(DEVICE), y.to(DEVICE)
        y = y.float()

        optimizer.zero_grad()
        preds = fusion_model(ecg, meta)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(fusion_train_dl)

    # ---------------------
    # VALIDATION
    # ---------------------
    fusion_model.eval()
    val_loss = 0

    with torch.no_grad():
        for ecg, meta, y in fusion_val_dl:
            ecg, meta, y = ecg.to(DEVICE), meta.to(DEVICE), y.to(DEVICE)
            y = y.float()

            preds = fusion_model(ecg, meta)
            loss = criterion(preds, y)
            val_loss += loss.item()

    val_loss = val_loss / len(fusion_val_dl)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # ---------------------
    # EARLY STOPPING LOGIC
    # ---------------------
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0

        # Save best model
        torch.save(fusion_model.state_dict(), "../outputs/best_fusion_model.pt")
        print("✅ Best model saved")

    else:
        epochs_no_improve += 1
        print(f"⚠️ No improvement ({epochs_no_improve}/{PATIENCE})")

        if epochs_no_improve >= PATIENCE:
            print("🛑 Early stopping triggered")
            break

Epoch 1/20 | Train Loss: 0.5143 | Val Loss: 0.3804
✅ Best model saved
Epoch 2/20 | Train Loss: 0.3308 | Val Loss: 0.3165
✅ Best model saved
Epoch 3/20 | Train Loss: 0.2807 | Val Loss: 0.2972
✅ Best model saved
Epoch 4/20 | Train Loss: 0.2640 | Val Loss: 0.2966
✅ Best model saved
Epoch 5/20 | Train Loss: 0.2546 | Val Loss: 0.2867
✅ Best model saved


## 6. Training of the fusion model

In [7]:
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

criterion = nn.BCEWithLogitsLoss()  # multi-label
optimizer = torch.optim.Adam(fusion_model.parameters(), lr=1e-3)

def train_one_epoch(model, dataloader):
    model.train()
    total_loss = 0

    for ecg, meta, labels in dataloader:
        ecg = ecg.to(DEVICE)
        meta = meta.to(DEVICE)
        labels = labels.to(DEVICE).float()

        optimizer.zero_grad()

        outputs = model(ecg, meta)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

## 7. Evaluation of the fusion model

In [8]:
def evaluate(model, dataloader):
    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for ecg, meta, labels in dataloader:
            ecg = ecg.to(DEVICE)
            meta = meta.to(DEVICE)

            outputs = model(ecg, meta)
            probs = torch.sigmoid(outputs)

            all_probs.append(probs.cpu())
            all_labels.append(labels)

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    preds = (all_probs > 0.5).astype(int)

    # Metrics
    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, average="macro")

    try:
        auc = roc_auc_score(all_labels, all_probs, average="macro")
    except:
        auc = float("nan")

    return acc, f1, auc

## 8. Full training

In [9]:
# Step 1 --> Load PTB-XL

from utils.data_loader import load_ptbxl

X_train, X_val, X_test, y_train, y_val, y_test, classes = load_ptbxl(PATH)

# Step 2 --> Load metadata

import pandas as pd
from utils.data_loader import load_metadata

Y = pd.read_csv(PATH + "ptbxl_database.csv", index_col="ecg_id")

meta = load_metadata(Y)

# Split metadata SAME way as ECG
train_folds = list(range(1, 9))
val_fold = 9
test_fold = 10

meta_train = meta[Y.strat_fold.isin(train_folds)]
meta_val = meta[Y.strat_fold == val_fold]
meta_test = meta[Y.strat_fold == test_fold]

# Step 3 --> Create Fusion DataLoaders

from torch.utils.data import DataLoader
from utils.data_loader import FusionDataset

train_dataset = FusionDataset(X_train, meta_train, y_train)
val_dataset = FusionDataset(X_val, meta_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

#
ecg, meta, labels = next(iter(train_loader))

print("ECG:", ecg.shape)
print("Meta:", meta.shape)
print("Labels:", labels.shape)
#

#Step 4 --> Training loop with validation metrics
num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(fusion_model, train_loader)

    val_acc, val_f1, val_auc = evaluate(fusion_model, val_loader)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val Accuracy: {val_acc:.4f}")
    print(f"Val F1: {val_f1:.4f}")
    print(f"Val AUC: {val_auc:.4f}")
    print("-" * 40)

ECG: torch.Size([32, 12, 1000])
Meta: torch.Size([32, 10])
Labels: torch.Size([32, 5])
Epoch 1/10
Train Loss: 0.2443
Val Accuracy: 0.6202
Val F1: 0.7028
Val AUC: 0.9237
----------------------------------------
Epoch 2/10
Train Loss: 0.2405
Val Accuracy: 0.6193
Val F1: 0.7150
Val AUC: 0.9251
----------------------------------------
Epoch 3/10
Train Loss: 0.2375
Val Accuracy: 0.6180
Val F1: 0.6961
Val AUC: 0.9230
----------------------------------------
Epoch 4/10
Train Loss: 0.2387
Val Accuracy: 0.6225
Val F1: 0.7107
Val AUC: 0.9254
----------------------------------------
Epoch 5/10
Train Loss: 0.2384
Val Accuracy: 0.6225
Val F1: 0.7175
Val AUC: 0.9260
----------------------------------------
Epoch 6/10
Train Loss: 0.2379
Val Accuracy: 0.6138
Val F1: 0.6838
Val AUC: 0.9229
----------------------------------------
Epoch 7/10
Train Loss: 0.2361
Val Accuracy: 0.6235
Val F1: 0.7202
Val AUC: 0.9259
----------------------------------------
Epoch 8/10
Train Loss: 0.2379
Val Accuracy: 0.6221
V

In [10]:
import numpy as np
import torch

from utils import (
    get_predictions,
    compute_auc_per_class,
    compute_macro_auc,
    plot_auc_bar,
    plot_prediction_vs_truth,
    plot_confusion_matrices,
    compute_all_metrics,
)

# Create a prediction function for fusion

def get_fusion_predictions(model, dataloader, device="cpu"):
    model.eval()

    all_probs = []
    all_labels = []

    with torch.no_grad():
        for ecg, meta, labels in dataloader:
            ecg = ecg.to(device)
            meta = meta.to(device)
            labels = labels.to(device)

            outputs = model(ecg, meta)  # <-- fusion forward

            probs = torch.sigmoid(outputs)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    return all_probs, all_labels

# Generate predictions

all_probs, all_labels = get_fusion_predictions(
    fusion_model,
    fusion_test_dl,
    device=DEVICE
)

# Compute fusion metrics

print("\n=== FUSION MODEL RESULTS ===")

fusion_metrics = compute_all_metrics(all_labels, all_probs)

for k, v in fusion_metrics.items():
    print(f"{k}: {v:.4f}")


=== FUSION MODEL RESULTS ===
Accuracy: 0.6288
F1: 0.7260
Precision (PPV): 0.7795
Sensitivity (Recall): 0.6904
Specificity: 0.9277
MCC: 0.6937
AUC: 0.9204


In [11]:
# Comparison table (xresnet1d101 vs fusion model)
# -------------------------
# ECG-only model metrics
# -------------------------

# Create model exactly like during training
ecg_model = xresnet1d101(input_channels=12, num_classes=5).to(DEVICE)

# Load weights, ignore mismatches if you modified later
checkpoint = torch.load("../outputs/best_ecg_model.pt", map_location=DEVICE)
ecg_model.load_state_dict(checkpoint, strict=False)

ecg_model.eval()

from torch.utils.data import DataLoader
from utils.data_loader import PTBXL_Dataset

# ECG-only dataset
test_dataset = PTBXL_Dataset(X_test, y_test)
test_dl = DataLoader(test_dataset, batch_size=32)

# Now run predictions
all_ecg_probs, all_ecg_labels = get_predictions(ecg_model, test_dl, device=DEVICE)

print("\n=== ECG MODEL RESULTS ===")
ecg_model.load_state_dict(torch.load("../outputs/best_ecg_model.pt", map_location=DEVICE))
ecg_model.eval()

all_ecg_probs, all_ecg_labels = get_predictions(ecg_model, test_dl, device=DEVICE)
ecg_metrics = compute_all_metrics(all_ecg_labels, all_ecg_probs)

for k, v in ecg_metrics.items():
    print(f"{k}: {v:.4f}")


# -------------------------
# Comparison table
# -------------------------
print("\n=== MODEL COMPARISON ===")
for key in ecg_metrics.keys():
    print(f"{key:20s} | ECG: {ecg_metrics[key]:.4f} | Fusion: {fusion_metrics[key]:.4f}")

C:\Users\inaki\AppData\Local\Temp\ipykernel_11100\1251860348.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("../outputs/best_ecg_model.pt", map


=== ECG MODEL RESULTS ===


C:\Users\inaki\AppData\Local\Temp\ipykernel_11100\1251860348.py:26: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ecg_model.load_state_dict(torch.load("../outputs/best_ecg_m

Accuracy: 0.6128
F1: 0.7105
Precision (PPV): 0.7691
Sensitivity (Recall): 0.6715
Specificity: 0.9254
MCC: 0.6771
AUC: 0.9197

=== MODEL COMPARISON ===
Accuracy             | ECG: 0.6128 | Fusion: 0.6288
F1                   | ECG: 0.7105 | Fusion: 0.7260
Precision (PPV)      | ECG: 0.7691 | Fusion: 0.7795
Sensitivity (Recall) | ECG: 0.6715 | Fusion: 0.6904
Specificity          | ECG: 0.9254 | Fusion: 0.9277
MCC                  | ECG: 0.6771 | Fusion: 0.6937
AUC                  | ECG: 0.9197 | Fusion: 0.9204


Fusion model is slightly better overall, especially in sensitivity and AUC. On the other hand, using metadata improves model decision-making without compromising overall reliability.

## 9. Save best model

In [13]:
best_f1 = 0

for epoch in range(num_epochs):
    train_loss = train_one_epoch(fusion_model, train_loader)
    val_acc, val_f1, val_auc = evaluate(fusion_model, val_loader)

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(fusion_model.state_dict(), "../outputs/best_fusion_model.pt")

    print(f"Epoch {epoch+1}: F1={val_f1:.4f}, AUC={val_auc:.4f}")

Epoch 1: F1=0.7094, AUC=0.9262
Epoch 2: F1=0.7149, AUC=0.9270
Epoch 3: F1=0.7069, AUC=0.9259
Epoch 4: F1=0.7060, AUC=0.9272
Epoch 5: F1=0.6996, AUC=0.9255
Epoch 6: F1=0.7025, AUC=0.9265
Epoch 7: F1=0.7090, AUC=0.9264
Epoch 8: F1=0.7166, AUC=0.9267
Epoch 9: F1=0.7067, AUC=0.9268
Epoch 10: F1=0.7074, AUC=0.9269
